# Query Iceberg Tables via DuckDB + Polaris REST Catalog

Reads `demo.users` and `demo.transactions` written by the Flink job, using
DuckDB's native Iceberg REST catalog support connected to Apache Polaris.

**Prerequisites:** the Docker Compose stack must be running (`docker compose up -d`).

In [ ]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("INSTALL iceberg")
con.execute("LOAD httpfs")
con.execute("LOAD iceberg")

# S3 credentials — DuckDB uses these to sign its own S3 requests directly
con.execute("""
CREATE OR REPLACE SECRET minio (
    TYPE        s3,
    KEY_ID      'minioadmin',
    SECRET      'minioadmin',
    ENDPOINT    'minio:9000',
    URL_STYLE   'path',
    USE_SSL     false,
    REGION      'us-east-1'
)
""")

# ACCESS_DELEGATION_MODE 'none' tells DuckDB to use the secret above for S3
# rather than requesting credentials from Polaris (which uses IAM vending)
con.execute("""
ATTACH 'demo_lh' AS polaris (
    TYPE                   iceberg,
    ENDPOINT               'http://polaris:8181/api/catalog',
    AUTHORIZATION_TYPE     'oauth2',
    CLIENT_ID              'root',
    CLIENT_SECRET          's3cr3t',
    OAUTH2_SERVER_URI      'http://polaris:8181/api/catalog/v1/oauth/tokens',
    SCOPE                  'PRINCIPAL_ROLE:ALL',
    ACCESS_DELEGATION_MODE 'none'
)
""")

con.execute("SHOW ALL TABLES").df()

---
## Users

In [ ]:
# All users
con.execute("SELECT * FROM polaris.demo.users ORDER BY created_at desc").df()

In [ ]:
# Summary stats
con.execute("""
    SELECT
        count(*)                AS total_users,
        count(DISTINCT country) AS unique_countries,
        min(created_at)         AS earliest_signup,
        max(created_at)         AS latest_signup
    FROM polaris.demo.users
""").df()

In [ ]:
# Users per country
con.execute("""
    SELECT
        country,
        count(*) AS user_count
    FROM polaris.demo.users
    GROUP BY country
    ORDER BY user_count DESC
""").df()

---
## Transactions

In [ ]:
# Latest 20 transactions
con.execute("""
    SELECT *
    FROM polaris.demo.transactions
    ORDER BY event_time DESC
    LIMIT 20
""").df()

In [ ]:
# Summary stats
con.execute("""
    SELECT
        count(*)                AS total_transactions,
        count(DISTINCT user_id) AS unique_users,
        round(sum(amount), 2)   AS total_volume,
        round(avg(amount), 2)   AS avg_amount,
        min(event_time)         AS earliest,
        max(event_time)         AS latest
    FROM polaris.demo.transactions
""").df()

In [ ]:
# Volume by transaction type
con.execute("""
    SELECT
        type,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM polaris.demo.transactions
    GROUP BY type
    ORDER BY tx_count DESC
""").df()

In [ ]:
# Breakdown by status
con.execute("""
    SELECT
        status,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM polaris.demo.transactions
    GROUP BY status
    ORDER BY tx_count DESC
""").df()

In [ ]:
# Volume by currency
con.execute("""
    SELECT
        currency,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_amount
    FROM polaris.demo.transactions
    GROUP BY currency
    ORDER BY total_amount DESC
""").df()

In [ ]:
# Top 10 users by transaction volume
con.execute("""
    SELECT
        user_id,
        count(*)              AS tx_count,
        round(sum(amount), 2) AS total_spent
    FROM polaris.demo.transactions
    GROUP BY user_id
    ORDER BY total_spent DESC
    LIMIT 10
""").df()

---
## Cross-table: join users ↔ transactions

In [ ]:
# Enrich transactions with user name and country
con.execute("""
    SELECT
        t.transaction_id,
        u.name,
        u.country,
        t.type,
        t.currency,
        t.amount,
        t.status,
        t.event_time
    FROM polaris.demo.transactions AS t
    JOIN polaris.demo.users        AS u ON t.user_id = u.user_id
    ORDER BY t.event_time DESC
    LIMIT 20
""").df()

In [ ]:
# Total transaction volume per country
con.execute("""
    SELECT
        u.country,
        count(*)                AS tx_count,
        round(sum(t.amount), 2) AS total_volume
    FROM polaris.demo.transactions AS t
    JOIN polaris.demo.users        AS u ON t.user_id = u.user_id
    GROUP BY u.country
    ORDER BY total_volume DESC
""").df()

---
## Snapshot history

In [ ]:
# Current snapshot info via DuckDB iceberg_snapshots()
for table_name in ("users", "transactions"):
    snap_df = con.execute(f"""
        SELECT snapshot_id, timestamp_ms, manifest_list
        FROM iceberg_snapshots('polaris.demo.{table_name}')
        ORDER BY timestamp_ms DESC
        LIMIT 1
    """).df()
    if not snap_df.empty:
        row = snap_df.iloc[0]
        print(f"=== {table_name} === snapshot_id={row.snapshot_id}  manifest_list={row.manifest_list}")
    else:
        print(f"=== {table_name} === no snapshots found")